In [1]:
!git clone -b megha-work https://github.com/meghagangal362015/cmpe257-early-reading-prediction.git

%cd cmpe257-early-reading-prediction

Cloning into 'cmpe257-early-reading-prediction'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 76 (delta 21), reused 54 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 1005.33 KiB | 6.09 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/cmpe257-early-reading-prediction


In [2]:
import pandas as pd

paper_df = pd.read_csv("data/processed/paper_complete_706.csv")

print("Dataset shape:", paper_df.shape)
print("Missing values:", paper_df.isna().sum().sum())

Dataset shape: (706, 19)
Missing values: 0


In [3]:
# Define the same 16 paper features, targets, and school groups

paper_features = [
    "NWFcls",
    "NWFwrc",
    "ORFwc",
    "SAwrS",
    "SAsrS",
    "SAtoS",
    "RMwidRS",
    "RMwdaRS",
    "Gender",
    "Tx",
    "Age1b",
    "Tier2_N",
    "TKPctCrct",
    "grp_rate",
    "rcmistot",
    "gnrl_fid"
]

X = paper_df[paper_features].copy()
groups = paper_df["SchlID"]

targets = {
    "Word Identification": "RMwidRS_target",
    "Word Attack": "RMwdaRS_target"
}

print("Feature matrix:", X.shape)
print("Number of schools:", groups.nunique())

Feature matrix: (706, 16)
Number of schools: 42


In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.base import clone
from xgboost import XGBClassifier
import numpy as np

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]),
        numeric_features
    )
])

group_kfold = GroupKFold(n_splits=5)

print("Numeric features:", numeric_features)
print("Five-fold grouped cross-validation is ready.")

Numeric features: ['NWFcls', 'NWFwrc', 'ORFwc', 'SAwrS', 'SAsrS', 'SAtoS', 'RMwidRS', 'RMwdaRS', 'Gender', 'Tx', 'Age1b', 'Tier2_N', 'TKPctCrct', 'grp_rate', 'rcmistot', 'gnrl_fid']
Five-fold grouped cross-validation is ready.


In [5]:
xgb_results = []

for target_name, target_column in targets.items():

    y = paper_df[target_column].astype(int)

    print(f"\n{target_name}")

    for fold, (train_index, validation_index) in enumerate(
        group_kfold.split(X, y, groups=groups),
        start=1
    ):
        X_train = X.iloc[train_index]
        X_validation = X.iloc[validation_index]

        y_train = y.iloc[train_index]
        y_validation = y.iloc[validation_index]

        model = Pipeline([
            ("preprocessor", clone(preprocessor)),
            (
                "classifier",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=3,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_validation)[:, 1]
        predictions = model.predict(X_validation)

        accuracy = accuracy_score(y_validation, predictions)
        pr_auc = average_precision_score(
            y_validation,
            probabilities
        )

        xgb_results.append({
            "Target": target_name,
            "Fold": fold,
            "Accuracy": accuracy,
            "PR-AUC": pr_auc
        })

        print(
            f"Fold {fold}: "
            f"Accuracy = {accuracy:.4f}, "
            f"PR-AUC = {pr_auc:.4f}"
        )


Word Identification
Fold 1: Accuracy = 0.6056, PR-AUC = 0.7074
Fold 2: Accuracy = 0.6056, PR-AUC = 0.6539
Fold 3: Accuracy = 0.6857, PR-AUC = 0.8463
Fold 4: Accuracy = 0.6000, PR-AUC = 0.5721
Fold 5: Accuracy = 0.5915, PR-AUC = 0.6379

Word Attack
Fold 1: Accuracy = 0.5211, PR-AUC = 0.6074
Fold 2: Accuracy = 0.6690, PR-AUC = 0.7042
Fold 3: Accuracy = 0.6000, PR-AUC = 0.8014
Fold 4: Accuracy = 0.5857, PR-AUC = 0.5447
Fold 5: Accuracy = 0.6972, PR-AUC = 0.7354


In [6]:
xgb_results_df = pd.DataFrame(xgb_results)

xgb_summary = (
    xgb_results_df
    .groupby("Target")[["Accuracy", "PR-AUC"]]
    .agg(["mean", "std"])
    .round(4)
)

print("XGBoost five-fold cross-validation results:")
display(xgb_summary)

print("\nIndividual fold results:")
display(xgb_results_df)

XGBoost five-fold cross-validation results:


Accuracy          PR-AUC        
                        mean     std    mean     std
Target                                              
Word Attack           0.6146  0.0699  0.6786  0.1024
Word Identification   0.6177  0.0385  0.6835  0.1030


Individual fold results:


,Target,Fold,Accuracy,PR-AUC
0,Word Identification,1,0.605634,0.707398
1,Word Identification,2,0.605634,0.653924
2,Word Identification,3,0.685714,0.846340
3,Word Identification,4,0.600000,0.572065
4,Word Identification,5,0.591549,0.637859
5,Word Attack,1,0.521127,0.607409
6,Word Attack,2,0.669014,0.704177
7,Word Attack,3,0.600000,0.801418
8,Word Attack,4,0.585714,0.544714
9,Word Attack,5,0.697183,0.735387


In [7]:
import os

os.makedirs("results", exist_ok=True)

xgb_results_df.to_csv(
    "results/xgboost_cv_fold_results.csv",
    index=False
)

xgb_summary.to_csv(
    "results/xgboost_cv_summary.csv"
)

print("XGBoost results saved.")

XGBoost results saved.


In [8]:
from google.colab import files

files.download("results/xgboost_cv_fold_results.csv")
files.download("results/xgboost_cv_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>